# Demo 09 : What each pattern costs

Run the cells in order and watch. There is nothing to write.

Run the next cell first. It loads the course key and builds `llm`, the course model. Nothing below it depends on where the notebook runs.

In [1]:
# Course setup : run this cell first.
# On the classroom laptop the key comes from .env in the repository root.
# In Colab, add it under the key icon in the left sidebar as COURSE_API_KEY.
import os
import sys
import subprocess
import warnings
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
COURSE_FILES = "https://raw.githubusercontent.com/FeikoWielsma/Multi-Agents-LangGraph-Student/main/"
ROOT = None

if IN_COLAB:
    # Pinned to match uv.lock ; generated by build/stamp_setup.py
    PINS = ["langchain[mcp]==1.4.6", "langchain-core==1.6.12", "langchain-openai==1.7.1", "langgraph==1.2.17", "langgraph-checkpoint-sqlite==3.1.2", "deepagents[quickjs]==0.7.30", "langchain-quickjs==0.3.9", "langsmith==0.14.12", "openai==3.29.0", "python-dotenv==1.2.4", "langchain-huggingface==1.2.3", "sentence-transformers==6.1.0", "langchain-text-splitters==1.1.3", "rank-bm25==0.2.2", "agentevals==0.0.9", "httpx==0.28.1", "pytest==9.1.1", "pandas==3.0.6"]
    if not os.getenv("COURSE_SKIP_INSTALL"):
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PINS], check=True)
    try:
        from google.colab import userdata
        os.environ.setdefault("COURSE_API_KEY", userdata.get("COURSE_API_KEY"))
    except Exception:
        pass
else:
    from dotenv import load_dotenv
    for folder in [Path.cwd(), *Path.cwd().parents]:
        if (folder / "pyproject.toml").exists():
            ROOT = folder
            load_dotenv(folder / ".env")
            break

PROVIDER = os.getenv("COURSE_PROVIDER", "course")      # "course" or "ollama"
if PROVIDER == "ollama":
    MODEL = os.getenv("COURSE_MODEL", "qwen3.5:4b")
    BASE_URL = "http://localhost:11434/v1"
    API_KEY = "ollama"
    THINKING_OFF = {"reasoning_effort": "none"}
else:
    MODEL = os.getenv("COURSE_MODEL", "openai/gpt-6-luna")  # xiaomi/mimo-v2.6-pro is the step up
    BASE_URL = os.getenv("COURSE_BASE_URL", "https://openrouter.ai/api/v1")
    API_KEY = os.getenv("COURSE_API_KEY")
    # OpenRouter has no Luna endpoint that honours temperature or parallel_tool_calls ;
    # seed is honoured, and routing prefers OpenAI so repeated runs hit one provider
    THINKING_OFF = {"reasoning": {"effort": "none"}, "provider": {"order": ["openai"]}}
    if not API_KEY:
        raise RuntimeError("COURSE_API_KEY is missing : copy .env.example to .env and paste the key")

from langchain_core._api import LangChainBetaWarning
warnings.filterwarnings("ignore", category=LangChainBetaWarning)

from langchain_openai import ChatOpenAI
from langchain_core.rate_limiters import InMemoryRateLimiter

# One shared limiter paces every model call in this process ; COURSE_RPS sets the budget.
# Paid OpenRouter models have no platform request cap (measured 26 October 2026)
RATE_LIMITER = InMemoryRateLimiter(requests_per_second=float(os.getenv("COURSE_RPS", "2")),
                                   check_every_n_seconds=0.1, max_bucket_size=1)


def make_llm(**kw):
    """The course chat model. Keyword arguments pass straight to ChatOpenAI."""
    kw.setdefault("model", MODEL)
    if PROVIDER == "ollama":
        kw.setdefault("temperature", 0)
    else:
        kw.setdefault("seed", 7)
    kw.setdefault("extra_body", THINKING_OFF)
    kw.setdefault("timeout", 90)        # seconds per request ; a stuck call raises instead of hanging
    kw.setdefault("rate_limiter", RATE_LIMITER)
    return ChatOpenAI(base_url=BASE_URL, api_key=API_KEY, **kw)


def make_embeddings(**kw):
    """Local sentence embeddings, so retrieval costs no API calls."""
    from langchain_huggingface import HuggingFaceEmbeddings
    kw.setdefault("model_name", "sentence-transformers/all-MiniLM-L6-v2")
    return HuggingFaceEmbeddings(**kw)


def data(name):
    """Path to a course data file ; downloaded when there is no local copy."""
    if ROOT and (ROOT / "data" / name).exists():
        return str(ROOT / "data" / name)
    target = Path(name)
    if not target.exists():
        import urllib.request
        target.parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(COURSE_FILES + "data/" + name, target)
    return str(target)


def data_folder(name):
    """Path to a course data folder ; downloaded file by file, using its index.txt, when there is no local copy."""
    if ROOT and (ROOT / "data" / name).is_dir():
        return str(ROOT / "data" / name)
    for file in Path(data(f"{name}/index.txt")).read_text(encoding="utf-8").split():
        data(f"{name}/{file}")
    return name


llm = make_llm()
print(f"model {MODEL} via {PROVIDER}")

model openai/gpt-6-luna via course


We use the made-up Veenhaven Bus records. We count every attempted model call, input and output tokens, and elapsed seconds, which include the wait in the shared rate limiter.

In [2]:
import json
import operator
import re
import sqlite3
import threading
import time
from pathlib import Path
from typing import Annotated, Literal
from typing_extensions import NotRequired, TypedDict

from langchain.agents import AgentState, create_agent
from langchain.agents.middleware import wrap_model_call
from langchain.messages import AIMessage, HumanMessage, SystemMessage, ToolMessage
from langchain.tools import ToolRuntime, tool
from langchain_core.callbacks import BaseCallbackHandler
from langgraph.graph import END, START, MessagesState, StateGraph
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.types import Command, Send
from pydantic import BaseModel, Field


class Counter(BaseCallbackHandler):
    def __init__(self):
        self.lock = threading.Lock()
        self.reset()

    def reset(self):
        self.calls = self.input = self.output = self.invalid = self.errors = 0

    def on_chat_model_start(self, serialized, messages, **kwargs):
        with self.lock:
            self.calls += 1

    def on_llm_end(self, response, **kwargs):
        with self.lock:
            for generations in response.generations:
                for generation in generations:
                    message = getattr(generation, "message", None)
                    usage = getattr(message, "usage_metadata", None) or {}
                    self.input += usage.get("input_tokens", 0)
                    self.output += usage.get("output_tokens", 0)
                    self.invalid += len(getattr(message, "invalid_tool_calls", []) or [])

    def on_llm_error(self, error, **kwargs):
        with self.lock:
            self.errors += 1


counter = Counter()
llm = make_llm(callbacks=[counter], max_tokens=1200, max_retries=0)
metrics = []


def measured(label, operation):
    counter.reset()
    start = time.perf_counter()
    try:
        return operation()
    finally:
        row = dict(pattern=label, calls=counter.calls, input_tokens=counter.input,
                   output_tokens=counter.output, seconds=round(time.perf_counter() - start, 1),
                   malformed=counter.invalid, errors=counter.errors)
        metrics.append(row)
        print("metrics :", json.dumps(row))


DB = Path(data("demo-world/service.db")).resolve()
MANUALS = Path(data_folder("demo-world/manuals"))
PAGES = {p.name: p.read_text(encoding="utf-8") for p in sorted(MANUALS.glob("*.md"))}


@tool
def part_stock(part: str) -> str:
    """Look up the stock, location and name of a Veenhaven Bus part number."""
    with sqlite3.connect(f"file:{DB.as_posix()}?mode=ro", uri=True) as db:
        rows = db.execute("SELECT part_no, name, stock, depot FROM parts WHERE part_no = ?",
                          (part.upper(),)).fetchall()
    return json.dumps(rows)


@tool
def fault_guidance(error_code: str) -> str:
    """Read the manual's meaning and first action for a Veenhaven Bus error code."""
    matches = [f"[{name}]\n{line}" for name, text in PAGES.items()
               for line in text.splitlines() if line.startswith("| " + error_code.upper() + " |")]
    return "\n".join(matches) or "No matching error code."


@tool
def run_sql(query: str) -> str:
    """Run a read-only SELECT. Tables: buses(fleet_no, model, depot_id, commissioned, firmware),
    depots(depot_id, name, city, contract), parts(part_no, model, name, stock, price_eur, depot),
    work_orders(wo_id, fleet_no, opened, status, error_code, summary, mechanic_id)."""
    if not query.lstrip().lower().startswith("select"):
        return "Only SELECT statements are allowed."
    with sqlite3.connect(f"file:{DB.as_posix()}?mode=ro", uri=True) as db:
        return json.dumps(db.execute(query).fetchmany(30))


@tool
def search_manuals(query: str) -> str:
    """Find relevant Veenhaven Bus manuals and bulletins with their file names."""
    words = set(query.lower().split())
    ranked = sorted(PAGES, key=lambda name: -sum(PAGES[name].lower().count(w) for w in words))[:3]
    return "\n\n".join(f"[{name}]\n{PAGES[name]}" for name in ranked)


QUESTION = ("Bus CE12-2021-0117 shows F-CE-110. How many CE-CF-02 cooling fans are in stock, "
            "at which location, and what are the manual's first action and replacement condition for this error? Be brief.")
STOCK_TASK = "How many CE-CF-02 cooling fans are in stock and at which location?"
MANUAL_TASK = "What are the meaning, first action and replacement condition for F-CE-110? Cite the manual."
LIMIT = {"recursion_limit": 24}


def ask(question):
    return {"messages": [HumanMessage(question)]}


def last_text(result):
    return str(next(m.content for m in reversed(result["messages"])
                    if isinstance(m, AIMessage) and m.content and not m.tool_calls))


def agent_by_hand(model, tools, prompt, state_schema=MessagesState):
    bound = model.bind_tools(tools)

    def model_node(state):
        return {"messages": [bound.invoke([SystemMessage(prompt), *state["messages"]])]}

    graph = StateGraph(state_schema)
    graph.add_node("model", model_node)
    graph.add_node("tools", ToolNode(tools))
    graph.add_edge(START, "model")
    graph.add_conditional_edges("model", tools_condition)
    graph.add_edge("tools", "model")
    return graph.compile()


def specialists(model):
    return {
        "parts": agent_by_hand(model, [part_stock], "Answer stock questions from the tool. Return every fact briefly."),
        "manuals": agent_by_hand(model, [fault_guidance], "Answer error-code questions from the manual tool. Cite its file."),
    }


def compose(model, question, findings):
    return model.invoke("Answer from these findings only. Preserve counts, locations and conditions.\n"
                        + question + "\n" + "\n".join(findings)).content


def completeness(answer):
    text = answer.lower()
    return {"stock": bool(re.search(r"\b(6|six)\b", text)) and "rietpoort" in text,
            "action": "fan" in text and "airflow" in text and bool(re.search(r"\b(80|eighty)\b", text))}

## a · what-each-pattern-costs

We run one question three times per pattern on the real demo records. We include one agent first, the five documented patterns, and a code-driven lookup variant. We compare the median of each column and inspect every answer. Seconds include the wait in the shared rate limiter at the configured COURSE_RPS.

In [3]:
workers = specialists(llm)
inner = {}


@tool
def call_parts(question: str) -> str:
    """Ask the parts specialist about stock and location."""
    result = workers["parts"].invoke(ask(question), LIMIT)
    inner.setdefault("parts", []).append(result["messages"])
    return last_text(result)


@tool
def call_manuals(question: str) -> str:
    """Ask the manuals specialist about error meanings and first actions."""
    result = workers["manuals"].invoke(ask(question), LIMIT)
    inner.setdefault("manuals", []).append(result["messages"])
    return last_text(result)


coordinator = agent_by_hand(llm, [call_parts, call_manuals],
                           "Coordinate Veenhaven Bus maintenance. Delegate both data questions, then answer briefly.")

In [4]:
class Route(BaseModel):
    agent: Literal["parts", "manuals"]
    query: str


class Routes(BaseModel):
    routes: list[Route] = Field(min_length=1, max_length=2)


class RouterState(TypedDict):
    question: str
    routes: list[dict]
    answers: Annotated[list[str], operator.add]
    final: str


def router_graph(model, parallel):
    workers = specialists(model)

    def classify(state):
        routes = model.with_structured_output(Routes).invoke(
            "Choose the required specialists. parts handles stock, manuals handles error codes. "
            "At most one question per specialist.\n" + state["question"]).routes
        print("routes :", [(r.agent, r.query) for r in routes])
        return {"routes": [r.model_dump() for r in routes]}

    def dispatch(state):
        return [Send(r["agent"], {"question": r["query"]}) for r in state["routes"]]

    def specialist(name):
        return lambda s: {"answers": [name + " : " + last_text(workers[name].invoke(ask(s["question"]), LIMIT))]}

    def sequential_dispatch(state):
        return {"answers": [r["agent"] + " : " + last_text(workers[r["agent"]].invoke(ask(r["query"]), LIMIT))
                            for r in state["routes"]]}

    graph = StateGraph(RouterState)
    graph.add_node("classify", classify)
    graph.add_node("synthesise", lambda s: {"final": compose(model, s["question"], s["answers"])})
    graph.add_edge(START, "classify")
    if parallel:
        for name in workers:
            graph.add_node(name, specialist(name))
            graph.add_edge(name, "synthesise")
        graph.add_conditional_edges("classify", dispatch, ["parts", "manuals"])
    else:
        graph.add_node("dispatch", sequential_dispatch)
        graph.add_edge("classify", "dispatch")
        graph.add_edge("dispatch", "synthesise")
    graph.add_edge("synthesise", END)
    return graph.compile()

In [5]:
class HandoffState(AgentState):
    active_agent: NotRequired[str]


refused = []


def transfer(target, fixed):
    @tool("transfer_to_" + target)
    def move(runtime: ToolRuntime) -> Command | str:
        """Transfer the conversation to the other specialist. Call it on its own, after your lookups."""
        messages = runtime.state["messages"]
        if fixed and len(messages[-1].tool_calls) > 1:
            refused.append(target)
            return "Transfer refused : call the transfer on its own, after the other tool results."
        note = ToolMessage(content="Transferred to " + target, tool_call_id=runtime.tool_call_id)
        forwarded = messages if fixed else [next(m for m in reversed(messages) if isinstance(m, AIMessage))]
        return Command(goto=target, graph=Command.PARENT,
                       update={"active_agent": target, "messages": [*forwarded, note]})
    return move


def handoff_graph(model, fixed=True):
    parts = agent_by_hand(model, [part_stock, transfer("manuals_agent", fixed)],
                          "Look up the stock first. Then transfer to manuals_agent if the error guidance is unanswered.", HandoffState)
    manuals = agent_by_hand(model, [fault_guidance, transfer("parts_agent", fixed)],
                            "Look up error guidance. Transfer only if stock is still unanswered. "
                            "When both facts are present, give one complete brief answer.", HandoffState)

    def route(state):
        last = state["messages"][-1]
        return END if isinstance(last, AIMessage) and not last.tool_calls else state["active_agent"]

    graph = StateGraph(HandoffState)
    graph.add_node("parts_agent", lambda s: parts.invoke(s, LIMIT))
    graph.add_node("manuals_agent", lambda s: manuals.invoke(s, LIMIT))
    graph.add_conditional_edges(START, lambda s: s.get("active_agent", "parts_agent"), ["parts_agent", "manuals_agent"])
    for name in ["parts_agent", "manuals_agent"]:
        graph.add_conditional_edges(name, route, ["parts_agent", "manuals_agent", END])
    return graph.compile()

In [6]:
SKILLS = {
    "parts": "Use part_stock to report the part number, count and location. Do not promise a reservation.",
    "manuals": "Use fault_guidance. Cite the file, preserve conditions, and distinguish inspection from replacement.",
}


@tool
def load_skill(name: str) -> str:
    """Load task instructions. Names: parts for stock; manuals for error-code guidance."""
    return SKILLS.get(name, "Unknown skill.")


skilled = agent_by_hand(llm, [load_skill, part_stock, fault_guidance],
                       "You support Veenhaven Bus. Load each matching skill before using its tools. Be brief.")

In [7]:
class WorkflowState(TypedDict):
    question: str
    stock: str
    guidance: str
    answer: str


def workflow_graph(model, parallel):
    workers = specialists(model)
    graph = StateGraph(WorkflowState)
    graph.add_node("parts", lambda s: {"stock": last_text(workers["parts"].invoke(ask(STOCK_TASK), LIMIT))})
    graph.add_node("manuals", lambda s: {"guidance": last_text(workers["manuals"].invoke(ask(MANUAL_TASK), LIMIT))})
    graph.add_node("compose", lambda s: {"answer": compose(model, s["question"], [s["stock"], s["guidance"]])})
    graph.add_edge(START, "parts")
    if parallel:
        graph.add_edge(START, "manuals")
        graph.add_edge(["parts", "manuals"], "compose")
    else:
        graph.add_edge("parts", "manuals")
        graph.add_edge("manuals", "compose")
    graph.add_edge("compose", END)
    return graph.compile()

In [8]:
single = agent_by_hand(llm, [part_stock, fault_guidance], "Answer briefly using the data tools.")
patterns = {
    "single agent": lambda: last_text(single.invoke(ask(QUESTION), LIMIT)),
    "subagents": lambda: last_text(coordinator.invoke(ask(QUESTION), LIMIT)),
    "handoffs fixed": lambda: last_text(handoff_graph(llm).invoke(ask(QUESTION), LIMIT)),
    "skills": lambda: last_text(skilled.invoke(ask(QUESTION), LIMIT)),
    "router": lambda: router_graph(llm, True).invoke({"question": QUESTION, "answers": []}, LIMIT)["final"],
    "custom workflow": lambda: workflow_graph(llm, False).invoke({"question": QUESTION}, LIMIT)["answer"],
    "code lookups": lambda: compose(llm, QUESTION, [part_stock.invoke({"part": "CE-CF-02"}), fault_guidance.invoke({"error_code": "F-CE-110"})]),
}
rows = []
refused.clear()
for name, operation in patterns.items():
    for run in range(1, 4):
        answer = measured(name, operation)
        checks = completeness(answer)
        row = {**metrics[-1], "run": run, "model": MODEL, "rps": float(os.environ["COURSE_RPS"]),
               "checks": checks, "correct": all(checks.values()), "answer": answer}
        rows.append(row)
        print("run :", name, run, "checks :", checks)
        print("answer :", answer)

from statistics import median

print("pattern              correct  calls  input  output  seconds")
for name in patterns:
    runs = [r for r in rows if r["pattern"] == name]
    values = [median(r[key] for r in runs) for key in ["calls", "input_tokens", "output_tokens", "seconds"]]
    print(f"{name:20} {sum(r['correct'] for r in runs)}/3      {values[0]:4.0f} {values[1]:6.0f} {values[2]:6.0f} {values[3]:7.1f}")
print("refused transfers in the handoff runs :", len(refused))
_ = Path("demo09-results.json").write_text(json.dumps(rows, indent=2), encoding="utf-8")

metrics : {"pattern": "single agent", "calls": 2, "input_tokens": 435, "output_tokens": 109, "seconds": 6.2, "malformed": 0, "errors": 0}
run : single agent 1 checks : {'stock': True, 'action': True}
answer : CE-CF-02 cooling fans: **6 in stock at Rietpoort**. For F-CE-110, first **check the traction cooling fan**; replace it if airflow is **below 80%**.


metrics : {"pattern": "single agent", "calls": 2, "input_tokens": 435, "output_tokens": 120, "seconds": 5.3, "malformed": 0, "errors": 0}
run : single agent 2 checks : {'stock': True, 'action': True}
answer : There are **6 CE-CF-02 traction cooling fans** in stock at **Rietpoort**. For **F-CE-110** (traction system over-temperature), first **check the traction cooling fan**; **replace it if airflow is below 80%**.


metrics : {"pattern": "single agent", "calls": 2, "input_tokens": 435, "output_tokens": 123, "seconds": 5.0, "malformed": 0, "errors": 0}
run : single agent 3 checks : {'stock': True, 'action': True}
answer : There are **6 CE-CF-02 traction cooling fans** in stock at **Rietpoort**. For **F-CE-110** (traction system over-temperature), the manual says to **check the traction cooling fan** and **replace it if airflow is below 80%**.


metrics : {"pattern": "subagents", "calls": 6, "input_tokens": 996, "output_tokens": 281, "seconds": 15.5, "malformed": 0, "errors": 0}
run : subagents 1 checks : {'stock': True, 'action': True}
answer : There are **6 CE-CF-02 cooling fans** in stock at **Rietpoort**. For **F-CE-110**, first check the traction cooling fan; **replace it if airflow is below 80%**.


metrics : {"pattern": "subagents", "calls": 6, "input_tokens": 942, "output_tokens": 260, "seconds": 14.8, "malformed": 0, "errors": 0}
run : subagents 2 checks : {'stock': True, 'action': True}
answer : There are **6 CE-CF-02 fans** in stock at **Rietpoort**. For **F-CE-110**, first **check the traction cooling fan**; replace it if airflow is **below 80%**.


...

metrics : {"pattern": "custom workflow", "calls": 5, "input_tokens": 633, "output_tokens": 191, "seconds": 12.8, "malformed": 0, "errors": 0}
run : custom workflow 2 checks : {'stock': True, 'action': True}
answer : There are **6 CE-CF-02 cooling fans** in stock at **Rietpoort**. For **F-CE-110**, the manual’s first action is to **check the traction cooling fan**; replace it if airflow is **below 80%**.


metrics : {"pattern": "custom workflow", "calls": 5, "input_tokens": 632, "output_tokens": 181, "seconds": 11.6, "malformed": 0, "errors": 0}
run : custom workflow 3 checks : {'stock': True, 'action': True}
answer : 6 CE-CF-02 cooling fans are in stock at **Rietpoort**. For **F-CE-110**, first **check the traction cooling fan**; replace it if airflow is **below 80%**.


metrics : {"pattern": "code lookups", "calls": 1, "input_tokens": 129, "output_tokens": 45, "seconds": 2.6, "malformed": 0, "errors": 0}
run : code lookups 1 checks : {'stock': True, 'action': True}
answer : 6 CE-CF-02 traction cooling fans are in stock at Rietpoort. For F-CE-110, first check the traction cooling fan; replace it if airflow is below 80 percent.


metrics : {"pattern": "code lookups", "calls": 1, "input_tokens": 129, "output_tokens": 57, "seconds": 2.8, "malformed": 0, "errors": 0}
run : code lookups 2 checks : {'stock': True, 'action': True}
answer : There are **6 CE-CF-02 traction cooling fans** in stock at **Rietpoort**. For F-CE-110, the manual says to **check the traction cooling fan** and replace it if airflow is **below 80 percent**.


metrics : {"pattern": "code lookups", "calls": 1, "input_tokens": 129, "output_tokens": 47, "seconds": 2.6, "malformed": 0, "errors": 0}
run : code lookups 3 checks : {'stock': True, 'action': True}
answer : There are 6 CE-CF-02 traction cooling fans in stock at Rietpoort. For F-CE-110, first check the traction cooling fan; replace it if airflow is below 80 percent.
pattern              correct  calls  input  output  seconds
single agent         3/3         2    435    120     5.3
subagents            3/3         6    942    260    15.4
handoffs fixed       3/3         4    947    121    10.2
skills               3/3         4   1139    159    10.2
router               3/3         6    825    246    15.2
custom workflow      3/3         5    632    181    12.8
code lookups         3/3         1    129     47     2.6
refused transfers in the handoff runs : 0


We score count and location plus the manual's airflow condition. We also read the answers because the checks can accept a wrong sentence containing the expected words. We compare calls and tokens first. Elapsed time includes deliberate request spacing and cannot be presented as model latency.

We keep the earlier probe separate. It used a smaller in-memory stock and workshop fixture at a different rate budget. Its handoff rows include the fixed history transfer, yet one run stopped with a half answer. We do not rerun Deep Agents here ; module 08 compares the harness after the manual patterns.

## Things to try

Optional : we can change the question, inspect each context, and compare the answer checks with the call and token counts.